# 📊 Quantitative Dataset Analysis

This notebook conducts a quantitative analysis of the datasets in the project:
1. **DJI logs:** Authentic forensic flight logs partitioned into `train`, `val`, and `test` files by drone hardware model in the `consistent_dataset` folder.
2. **ESP32 logs:** Bluetooth Remote ID telemetry generated by stationary spoofing microcontrollers.
3. **Simulated flights:** Custom simulated spoofing telemetry trajectories.

In [ ]:
import os
import re
import hashlib
import json
import pandas as pd
from pathlib import Path

# Resolve relative project root securely
notebook_dir = Path('.').resolve()
if notebook_dir.name == 'supervised':
    PROJECT_ROOT = notebook_dir.parents[1]
elif notebook_dir.name == 'notebooks':
    PROJECT_ROOT = notebook_dir.parent
else:
    # If running from project root
    PROJECT_ROOT = notebook_dir
    
DATASET_PATH = PROJECT_ROOT / 'dataset'

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATASET_PATH}")

## 1. DJI Logs: Hardware-Based Partition Split (consistent_dataset)

All flights are verified to have a minimum length threshold of 100 samples.

In [ ]:
import re
import sys
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# Ensure project supervised module is in path
if 'PROJECT_ROOT' in globals():
    sys.path.insert(0, str(PROJECT_ROOT))
else:
    sys.path.insert(0, str(Path('..').resolve()))

from implement.utils.helper.features import resample_df

consistent_dir = DATASET_PATH / 'genuine_dji_flights' / 'consistent_dataset'
all_files = sorted(list(consistent_dir.glob('*.csv')))

records = []
for file_path in all_files:
    df = pd.read_csv(file_path)
    row_count = len(df)
    
    # Calculate resampled length (2Hz)
    df_res = resample_df(df, 'latitude', 'longitude', 'altitude', 'ground_speed', 'course', 'timestamp', step_size=0.5)
    resampled_count = len(df_res)
    
    stem = file_path.stem
    parts = stem.split('_flight_')
    drone_model = 'Unknown'
    if len(parts) > 1:
        model_part = parts[1]
        match = re.match(r'^\d+_(.+)$', model_part)
        if match:
            drone_model = match.group(1).upper()
            
    records.append({
        'Filename': file_path.name,
        'Drone Model': drone_model,
        'Raw Samples (Rows)': row_count,
        'Resampled Samples (2Hz)': resampled_count
    })

df_all = pd.DataFrame(records)

# Dynamically and deterministically split models 80/20 using seed 42
unique_models = sorted(list(df_all['Drone Model'].unique()))
train_models, test_models = train_test_split(unique_models, test_size=0.20, random_state=42)

df_all['Split'] = df_all['Drone Model'].apply(lambda m: 'Test' if m in test_models else 'Train')

train_df = df_all[df_all['Split'] == 'Train']
test_df = df_all[df_all['Split'] == 'Test']

total_raw = df_all['Raw Samples (Rows)'].sum()
total_res = df_all['Resampled Samples (2Hz)'].sum()
train_raw = train_df['Raw Samples (Rows)'].sum()
train_res = train_df['Resampled Samples (2Hz)'].sum()
test_raw = test_df['Raw Samples (Rows)'].sum()
test_res = test_df['Resampled Samples (2Hz)'].sum()

print(f"=== DJI DYNAMIC 80/20 DEVICE SPLIT SUMMARY ===")
print(f"Total DJI Raw Samples       : {total_raw:,}")
print(f"Total DJI Resampled (2Hz)   : {total_res:,}")
print(f"Train Split (Raw)           : {len(train_df)} files, {train_raw:,} samples ({train_raw/total_raw*100:.2f}%)")
print(f"Train Split (Resampled 2Hz) : {len(train_df)} files, {train_res:,} samples ({train_res/total_res*100:.2f}%)")
print(f"Test Split (Held-out Raw)   : {len(test_df)} files, {test_raw:,} samples ({test_raw/total_raw*100:.2f}%)")
print(f"Test Split (Held-out 2Hz)   : {len(test_df)} files, {test_res:,} samples ({test_res/total_res*100:.2f}%)\n")

print("=== DJI Train Files Details ===")
print(train_df.sort_values(by='Drone Model').to_string(index=False))

print("\n=== DJI Test Files Details ===")
print(test_df.sort_values(by='Drone Model').to_string(index=False))

## 2. ESP32 Logs: Bluetooth Remote ID Telemetry

The ESP32 dataset contains Bluetooth Remote ID records. We isolate the location-based telemetry messages and examine their sample sizes.

In [ ]:
import sys
import pandas as pd
from pathlib import Path

# Ensure project supervised module is in path
if 'PROJECT_ROOT' in globals():
    sys.path.insert(0, str(PROJECT_ROOT))
else:
    sys.path.insert(0, str(Path('..').resolve()))

from implement.utils.helper.features import resample_df

esp32_path = DATASET_PATH / 'hardware_spoofer' / 'esp32_source_telemetry.csv'
df_esp32 = pd.read_csv(esp32_path)

print(f"=== ESP32 Log Summary ===")
print(f"Total raw rows: {len(df_esp32):,}")

# Message type distribution
print("\nMessage Type Distribution:")
print(df_esp32['Message Type'].value_counts())

# Isolate Location messages (spoofed drone path)
df_esp32_loc = df_esp32[df_esp32['Message Type'] == 'Location'].copy()
df_esp32_loc['Timestamp'] = pd.to_datetime(df_esp32_loc['Timestamp'])
df_esp32_loc['time_sec'] = (df_esp32_loc['Timestamp'] - df_esp32_loc['Timestamp'].iloc[0]).dt.total_seconds()

# Cast telemetry fields to numeric to avoid interpolation type errors
for col in ['Latitide', 'Logitude', 'Altitude Geodetic', 'Speed Horizontal', 'Direction']:
    df_esp32_loc[col] = pd.to_numeric(df_esp32_loc[col], errors='coerce')

# Resample
df_res = resample_df(
    df_esp32_loc, 
    'Latitide', 'Logitude', 'Altitude Geodetic', 'Speed Horizontal', 'Direction', 'time_sec', 
    step_size=0.5
)

print(f"\nLocation messages count (Active Spoofing Telemetry Raw)  : {len(df_esp32_loc):,}")
print(f"Location messages count (Active Spoofing Telemetry 2Hz)  : {len(df_res):,}")

## 3. Simulated Telemetry Spoofing Flights

We parse the custom pre-generated simulated trajectory flights inside the `dataset/curated_flights/` folder and inspect their file distributions and sample sizes.

In [ ]:
import sys
import pandas as pd
from pathlib import Path

# Ensure project supervised module is in path
if 'PROJECT_ROOT' in globals():
    sys.path.insert(0, str(PROJECT_ROOT))
else:
    sys.path.insert(0, str(Path('..').resolve()))

from implement.utils.helper.features import resample_df

sim_dir = DATASET_PATH / 'curated_flights'
sim_files = sorted(list(sim_dir.glob('*.csv')))

sim_records = []
for file_path in sim_files:
    df_sim = pd.read_csv(file_path)
    
    # Resample to 2Hz
    df_res = resample_df(
        df_sim, 
        'latitude', 'longitude', 'altitude', 'speed_horizontal', 'direction', 'timestamp', 
        step_size=0.5
    )
    resampled_count = len(df_res)
    
    # Determine profile prefix from filename (e.g. 'easy_flight_1.csv' -> 'easy')
    prefix = 'unknown'
    stem = file_path.stem
    if '_flight_' in stem:
        prefix = stem.split('_flight_')[0]
        
    sim_records.append({
        'Filename': file_path.name,
        'Profile Class': prefix,
        'Raw Samples (Rows)': len(df_sim),
        'Resampled Samples (2Hz)': resampled_count
    })

df_sim_all = pd.DataFrame(sim_records)
df_sim_grouped = df_sim_all.groupby('Profile Class')[['Filename', 'Raw Samples (Rows)', 'Resampled Samples (2Hz)']].agg({
    'Filename': 'count',
    'Raw Samples (Rows)': 'sum',
    'Resampled Samples (2Hz)': 'sum'
}).rename(columns={
    'Filename': 'Flights Count', 
    'Raw Samples (Rows)': 'Total Raw Samples', 
    'Resampled Samples (2Hz)': 'Total Resampled Samples (2Hz)'
})

print("=== Simulated Dataset Summary by Profile Class ===")
print(df_sim_grouped.to_string())

print("\n=== Details of Simulated Telemetry Flights ===")
print(df_sim_all.to_string(index=False))

## 4. Quantitative Analysis of Engineered Features & Dataset Overlaps

We load the engineered/preprocessed feature files for DJI and ESP32/Sim to compute and compare their statistical profiles (Mean and Standard Deviation) across the point-wise telemetry features.

In [ ]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Add supervised package to path relative to project root
sys.path.insert(0, str(PROJECT_ROOT))

from implement.utils.helper import get_or_preprocess_genuine_dji_flights, get_or_preprocess_hardware_spoofer
from implement.utils.helper.features import SUPERVISED_FEATURES

# Load datasets
dji = get_or_preprocess_genuine_dji_flights()
esp_sim = get_or_preprocess_hardware_spoofer()

print("=== Feature Column Consistency Check ===")
for name, df in [('DJI Normal', dji), ('ESP32 + Sim Spoofed', esp_sim)]:
    missing = [f for f in SUPERVISED_FEATURES if f not in df.columns]
    print(f"{name}: rows={len(df):,}, missing={missing}")

print("\n=== Feature Mean & Std Comparison across Datasets ===")
stats = []
for f in SUPERVISED_FEATURES:
    stats.append({
        'Feature': f,
        'DJI Mean': dji[f].mean() if f in dji.columns else np.nan,
        'DJI Std': dji[f].std() if f in dji.columns else np.nan,
        'Spoofed Mean': esp_sim[f].mean() if f in esp_sim.columns else np.nan,
        'Spoofed Std': esp_sim[f].std() if f in esp_sim.columns else np.nan,
    })
df_stats = pd.DataFrame(stats)
print(df_stats.to_string(index=False))

# Save statistics summary to output folder
output_dir = PROJECT_ROOT / 'output' / 'plots_distribution'
output_dir.mkdir(parents=True, exist_ok=True)
df_stats.to_csv(output_dir / 'feature_statistics_comparison.csv', index=False)
print(f"\n✓ Saved feature statistics comparison to: {output_dir / 'feature_statistics_comparison.csv'}")